# 头戴双目的手部姿态

设备还没有。这一本按规格 §7.7 的会话目录，把 **HaMeR / WiLoR（或 CPU 上的 MediaPipe）→ 双目三角化 → 相机位姿 → 统一 episode → QC 和骨架叠加** 串起来。统一 JSON 和开放数据用的是同一份，后面的质检、覆盖、导出不用改。

默认只跑 **合成双目**（已知三维点投影到左右相机），打印三角化误差，单位是厘米，应当接近 0。不下载权重，也不下载 EgoDex。

## MANO 许可

HaMeR 和 WiLoR 都要 MANO 右手模型。MANO 是马普所的 **非商业学术许可**，不能再分发，也不能放进这个仓库。

1. 打开 https://mano.is.tue.mpg.de 注册并同意 non-commercial license。
2. 下载 `mano_v1_2.zip`，解压出 `MANO_RIGHT.pkl`。
3. `export MANO_MODEL_DIR=/绝对路径/mano`（目录里放 `MANO_RIGHT.pkl`）。

HaMeR 权重按 https://github.com/geopavlakos/hamer 的 README 放到仓库外，再 `export HAMER_CHECKPOINT=/绝对路径/hamer.ckpt`。本笔记本不会调用 `download_models`。

WiLoR 权重是 CC-BY-NC-ND，从 https://huggingface.co/spaces/rolpotamias/WiLoR 取 `detector.pt`、`wilor_final.ckpt`、`model_config.yaml`：

```
export WILOR_CHECKPOINT=/绝对路径/wilor_final.ckpt
export WILOR_DETECTOR=/绝对路径/detector.pt
export WILOR_CONFIG=/绝对路径/model_config.yaml
```

没有这些文件时，用下面的合成双目和 MediaPipe 回退。MediaPipe 不需要 MANO，它的三维不是公制相机系，尺度靠三角化。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyxiaobai/robot/blob/main/notebooks/headcam_hand_pose_colab.ipynb)


In [ ]:
%pip install -q numpy pyyaml pillow matplotlib


In [ ]:
import os
import subprocess
from pathlib import Path

root = Path.cwd()
if not (root / "scripts" / "convert_headcam.py").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/skyxiaobai/robot.git", "/content/robot",
    ])
    os.chdir("/content/robot")
print("repo", Path.cwd())


## 合成双目：三角化误差（厘米）

左右相机已校正，基线 8 cm，焦距 200 px。把已知的三维点投到两个像平面，再三角化回来。这一步不需要 GPU，也不需要 MANO。


In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path.cwd() / "scripts"))
from headcam.hand_pose import triangulate_pair

K = np.array([[200.0, 0.0, 160.0], [0.0, 200.0, 120.0], [0.0, 0.0, 1.0]])
calib = {
    "K_left": K,
    "K_right": K.copy(),
    "dist_left": np.zeros(5),
    "dist_right": np.zeros(5),
    "R": np.eye(3),
    "T": np.array([-0.08, 0.0, 0.0]),
}
truth = np.array([
    [0.02, -0.01, 0.45],
    [-0.03, 0.02, 0.70],
    [0.00, 0.00, 1.10],
    [0.05, 0.04, 0.55],
])

def project(points, rotation, translation):
    pixels = []
    for point in points:
        camera = rotation @ point + translation
        pixels.append([
            K[0, 0] * camera[0] / camera[2] + K[0, 2],
            K[1, 1] * camera[1] / camera[2] + K[1, 2],
        ])
    return np.asarray(pixels)

left = project(truth, np.eye(3), np.zeros(3))
right = project(truth, calib["R"], calib["T"])
recovered, confidence = triangulate_pair(left, right, calib)
err_cm = float(np.mean(np.linalg.norm(recovered - truth, axis=1)) * 100.0)
print("三角化平均误差 %.4f cm" % err_cm)
print("逐点置信度", np.round(confidence, 4))


## 会话目录 → 统一 episode → QC → 叠加

单目三维故意放大两倍。右目 2D 和标定把尺度收回来，SLAM 轨迹是单位阵，所以世界系关节应和真值重合。然后跑现有的 QC，并把骨架画在灰底上（没有真实 mp4）。


In [ ]:
import json
from pathlib import Path

import numpy as np

import convert_headcam
from egodata.qc import qc_episode
from egodata.schema import save_episode
from ego_visualize import write_overlay
from headcam.hand_pose import write_calibration_yaml

out_dir = Path("outputs/headcam_demo")
session = out_dir / "session"
session.mkdir(parents=True, exist_ok=True)
write_calibration_yaml(session / "calib.yaml", {
    "image_width": 320,
    "image_height": 240,
    "K_left": K,
    "K_right": K,
    "dist_left": np.zeros(5),
    "dist_right": np.zeros(5),
    "R": np.eye(3),
    "T": np.array([-0.08, 0.0, 0.0]),
})
stamps = [index / 30.0 for index in range(4)]
(session / "timestamps.csv").write_text(
    "frame_index,timestamp_s\n" + "\n".join("%d,%.8f" % (i, t) for i, t in enumerate(stamps)),
    encoding="utf-8",
)
(session / "metadata.json").write_text(json.dumps({
    "episode_id": "headcam/colab",
    "fps": 30,
    "task": "pick",
    "instruction": "拿起",
    "environment": "tabletop",
    "image_width": 320,
    "image_height": 240,
}), encoding="utf-8")
(session / "slam.tum").write_text(
    "\n".join("%.8f 0 0 0 0 0 0 1" % stamp for stamp in stamps) + "\n",
    encoding="utf-8",
)

def hand_at(wrist):
    wrist = np.asarray(wrist, dtype=float)
    joints = np.zeros((21, 3))
    for index in range(21):
        joints[index] = wrist + [0.01 * ((index % 5) - 2), 0.004 * (index % 3), 0.008 * (index // 7)]
    joints[0] = wrist
    joints[5] = wrist + [0.04, 0.0, 0.01]
    joints[17] = wrist + [-0.03, 0.012, 0.01]
    return joints

frames = []
truth_wrist = []
for index in range(4):
    left = hand_at([0.01 * index, 0.02, 0.60])
    right_hand = hand_at([-0.02 + 0.01 * index, -0.03, 0.70])
    truth_wrist.append(left[0])
    frames.append({
        "left": {
            "joints_cam": (left * 2.0).tolist(),
            "keypoints_2d": project(left, np.eye(3), np.zeros(3)).tolist(),
            "confidence": [0.2] * 21,
        },
        "right": {
            "joints_cam": (right_hand * 2.0).tolist(),
            "keypoints_2d": project(right_hand, np.eye(3), np.zeros(3)).tolist(),
            "confidence": [0.2] * 21,
        },
        "stereo_keypoints_right_view": {
            "left": project(left, calib["R"], calib["T"]).tolist(),
            "right": project(right_hand, calib["R"], calib["T"]).tolist(),
        },
    })
(session / "hands.json").write_text(json.dumps({"frames": frames}), encoding="utf-8")

episode = convert_headcam.build_episode(session)
json_path = out_dir / "episode.json"
save_episode(episode, json_path)
got = np.asarray(episode["hands"]["left"]["joints"], dtype=float)[:, 0]
err_cm = float(np.mean(np.linalg.norm(got - np.stack(truth_wrist), axis=1)) * 100.0)
print("校正后左手腕误差 %.4f cm" % err_cm)
print("尺度", episode["hand_pose"]["scale"]["left"])
print(qc_episode(episode))
overlay = write_overlay(episode, out_dir / "overlay.png", width=320)
print("overlay", overlay)


## GPU：HaMeR 或 WiLoR

下面两个开关默认关闭。打开之前先按文首把 MANO 和对应权重放在仓库外，并设好环境变量。安装不会把 `*.pkl` / `*.ckpt` 写进仓库。

官方 demo 如果比这里的适配器更合适，用 demo 导出 `hands.json`，再跑上一格的 `convert_headcam.py`。


In [ ]:
INSTALL_HAMER = False
INSTALL_WILOR = False

import os
import subprocess
import sys

import numpy as np

from headcam.hand_pose import MANO_LICENSE, get_backend, hamer_missing, wilor_missing

print(MANO_LICENSE)
print("---")
if INSTALL_HAMER:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/geopavlakos/hamer.git"])
    print("HaMeR 已安装。还需要 detectron2、ViTPose，以及 HAMER_CHECKPOINT / MANO_MODEL_DIR。")
if INSTALL_WILOR:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/warmshao/WiLoR-mini.git"])
    print("若官方包更合适：git clone https://github.com/rolpotamias/WiLoR 并按其 README 安装。")

print("HaMeR 缺少:", hamer_missing() or "无")
print("WiLoR 缺少:", wilor_missing() or "无")

# 公开的第一视角 mp4，或一条 EgoDex mp4。留空则跳过。
SAMPLE_VIDEO = os.environ.get("HEADCAM_SAMPLE_VIDEO", "")
BACKEND = os.environ.get("HEADCAM_BACKEND", "hamer")
if not SAMPLE_VIDEO:
    print("没有 SAMPLE_VIDEO。设 HEADCAM_SAMPLE_VIDEO 指向 mp4 后再跑单目估计。")
elif hamer_missing() and wilor_missing() and BACKEND != "mediapipe":
    print("权重或包还不齐，不把合成数字当成模型误差。")
else:
    import cv2
    backend = get_backend(BACKEND)
    capture = cv2.VideoCapture(SAMPLE_VIDEO)
    ok, frame = capture.read()
    capture.release()
    if not ok:
        raise RuntimeError("读不到视频帧：" + SAMPLE_VIDEO)
    prediction = backend.predict(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    for side in ("left", "right"):
        joints = prediction[side]["joints_cam"]
        print(side, "wrist_cam", None if joints is None else np.round(joints[0], 4))


## 和 EgoDex 手腕真值比（厘米）

EgoDex 的 mp4 是单目，真值在 ARKit 世界系。比较前把真值用该帧相机位姿变回相机系，再和后端的相机系手腕做 L2。这个数含单目尺度误差。双目公制误差看上面的合成单元。

没有 hdf5 时，用 1 cm 的平移检查这个指标本身，这个 1 cm **不是**模型误差。


In [ ]:
import os

import numpy as np

from headcam.hand_pose import get_backend, hamer_missing, mean_wrist_error_m, wilor_missing, world_to_camera

demo = np.zeros((8, 21, 3))
shifted = demo.copy()
shifted[:, 0, 0] += 0.01
print("指标自检：平移 1 cm 时手腕误差 %.2f cm（不是模型输出）" % (mean_wrist_error_m(shifted, demo) * 100.0))

EGODEX_HDF5 = os.environ.get("EGODEX_HDF5", "")
EGODEX_MP4 = os.environ.get("EGODEX_MP4", "")
BACKEND = os.environ.get("HEADCAM_BACKEND", "hamer")
MAX_FRAMES = 20

if not EGODEX_HDF5 or not EGODEX_MP4:
    print("未设置 EGODEX_HDF5 / EGODEX_MP4，跳过和真值的比较。")
elif BACKEND != "mediapipe" and ((BACKEND == "hamer" and hamer_missing()) or (BACKEND == "wilor" and wilor_missing())):
    print("后端还不齐。装好 MANO 和权重后再比，避免把上面的 1 cm 当成 HaMeR/WiLoR 的误差。")
else:
    import cv2
    from egodata.egodex import load_episode_hdf5

    episode = load_episode_hdf5(EGODEX_HDF5)
    backend = get_backend(BACKEND)
    capture = cv2.VideoCapture(EGODEX_MP4)
    estimated = []
    truth = []
    index = 0
    while index < min(MAX_FRAMES, episode["num_frames"]):
        ok, frame = capture.read()
        if not ok:
            break
        prediction = backend.predict(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        pose = np.asarray(episode["camera_poses"][index], dtype=float)
        for side in ("left", "right"):
            gt = episode["hands"][side]["joints"][index]
            est = prediction[side]["joints_cam"]
            if gt[0][0] is None or est is None:
                continue
            gt_cam = world_to_camera(np.asarray(gt, dtype=float), pose)
            estimated.append(est)
            truth.append(gt_cam)
        index += 1
    capture.release()
    if not estimated:
        print("这几帧里没有同时具备真值和预测的手。")
    else:
        err_m = mean_wrist_error_m(np.stack(estimated), np.stack(truth))
        print("%s 手腕平均误差 %.2f cm（%d 只手，单目，含尺度）" % (BACKEND, err_m * 100.0, len(estimated)))
